# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene1407/ML-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip install -q duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import duckdb
import getpass

# Connect to DuckDB
con = duckdb.connect()

# Enter your Hugging Face token securely
hf_token = getpass.getpass("Enter your Hugging Face Read token: ")

# Register authentication with DuckDB
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN ?
    )
""", [hf_token])

print("DuckDB connection created successfully!")

DuckDB connection created successfully!


In [3]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

schema = con.execute(f"DESCRIBE SELECT * FROM {REL}").df()

print(schema[["column_name", "column_type"]].to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature fields

These fields describe search performance during March 2026 and are available before the April outcome period.

- `gsc_impressions`: Number of search impressions during the observation period.
- `gsc_clicks`: Number of clicks received from search.
- `gsc_avg_position`: Observed average search position.
- `march_ctr`: Derived from March clicks and impressions.
- `gsc_active_days`: Number of March days with available GSC data.

### Label / Proxy

- `position_worsened`: A binary proxy indicating whether a content item's average search position worsened in April compared with March. A higher average position value means a worse ranking. This label is calculated using April data and must never be used as a feature.

### Context

- `report_date`: Identifies the reporting date and observation window.
- `client_hash_id`: Identifies the client for grouping and validation.
- `content_hash_id`: Identifies the content item for grouping and joining.
- `month`: Identifies the monthly partition.
- `gsc_data_available`: Identifies whether GSC measurements are available.

### Excluded

- `gsc_sum_position`: Excluded as a separate feature because average position is already represented by `gsc_avg_position`.
- GA4 metrics: Excluded from this GSC-focused analysis to keep the feature set consistent and avoid mixing measurement availability across clients.
- April performance fields: Excluded from features because they belong to the future outcome period.
- Client and content identifiers: Excluded from model features because they identify entities rather than describe their search performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {REL}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

duplicates = con.execute(grain_query).df()

print("Duplicate combinations found:", len(duplicates))
display(duplicates)

Duplicate combinations found: 0


,report_date,client_hash_id,content_hash_id,row_count


In [5]:
count_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {REL}
"""

march_summary = con.execute(count_query).df()
display(march_summary)

,total_rows,unique_clients,unique_content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [6]:
availability_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS unavailable_rows,
    COUNT(DISTINCT client_hash_id) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clients_with_gsc
FROM {REL}
"""

availability = con.execute(availability_query).df()
display(availability)

,total_rows,available_rows,unavailable_rows,clients_with_gsc
0,9841378,3611061,6230317,47


**1. Unit of Analysis**

One row in the warehouse represents one content item belonging to one client on one reporting date.

The unique grain is:
`report_date + client_hash_id + content_hash_id`

**2. Tables Used**

Primary table: `fact_content_daily_performance`

This table contains daily Google Search Console performance metrics and Google Analytics measurements.

**3. Time Window**

Development window: March 1–31, 2026.

I will use March as the feature observation period and April 2026 as the future outcome window, keeping June 2026 sealed for final testing.

**4. Prediction / Ranking Objective**

I want to investigate which content items are more likely to experience a deterioration in average search position in the following month.

The proposed proxy label is whether a content item's average search position worsens in April compared with March.

**5. Deliberate Exclusion**

I will exclude client and content identifiers from model features because they are identifiers used for grouping and joining, not meaningful predictive signals. I will also exclude unavailable GSC records rather than treating missing search data as zero.

In [7]:
MARCH = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

APRIL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

feature_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_avg_position * gsc_impressions)
            / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_avg_position * gsc_impressions)
            / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions,
    m.clicks,
    m.avg_position,
    m.clicks * 100.0 / NULLIF(m.impressions, 0) AS march_ctr,
    m.active_days,
    CASE
        WHEN a.avg_position > m.avg_position THEN 1
        ELSE 0
    END AS position_worsened
FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
"""

feature_df = con.execute(feature_query).df()

print("Feature frame shape:", feature_df.shape)
display(feature_df.head())

Feature frame shape: (158549, 8)


,client_hash_id,content_hash_id,impressions,clicks,avg_position,march_ctr,active_days,position_worsened
0,client_3ffa76342f366962,content_a1654ed76a182627,1.0,0.0,9.000000,0.000000,1,0
1,client_73cda7b4e4f265ea,content_6de123f6aca5ca67,1380.0,3.0,4.017391,0.217391,31,1
2,client_73cda7b4e4f265ea,content_6d2b2347d240c5c6,359.0,1.0,10.320334,0.278552,31,1
3,client_73cda7b4e4f265ea,content_6a979e7411fc9705,16606.0,129.0,2.396423,0.776828,31,1
4,client_73cda7b4e4f265ea,content_644a50a181473271,375.0,0.0,13.077333,0.000000,31,0


In [8]:
print("Feature frame shape:", feature_df.shape)

print("\nColumn names:")
print(feature_df.columns.tolist())

print("\nFirst 5 rows:")
display(feature_df.head())

print("\nMissing values:")
display(feature_df.isnull().sum())


Feature frame shape: (158549, 8)

Column names:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'march_ctr', 'active_days', 'position_worsened']

First 5 rows:


,client_hash_id,content_hash_id,impressions,clicks,avg_position,march_ctr,active_days,position_worsened
0,client_3ffa76342f366962,content_a1654ed76a182627,1.0,0.0,9.000000,0.000000,1,0
1,client_73cda7b4e4f265ea,content_6de123f6aca5ca67,1380.0,3.0,4.017391,0.217391,31,1
2,client_73cda7b4e4f265ea,content_6d2b2347d240c5c6,359.0,1.0,10.320334,0.278552,31,1
3,client_73cda7b4e4f265ea,content_6a979e7411fc9705,16606.0,129.0,2.396423,0.776828,31,1
4,client_73cda7b4e4f265ea,content_644a50a181473271,375.0,0.0,13.077333,0.000000,31,0



Missing values:


client_hash_id       0
content_hash_id      0
impressions          0
clicks               0
avg_position         0
march_ctr            0
active_days          0
position_worsened    0
dtype: int64

### Five Features and Their Availability

| Feature | Available when? |
|---|---|
| `impressions` | Available at the end of March because it is calculated from March GSC impressions. |
| `clicks` | Available at the end of March because it is calculated from March GSC clicks. |
| `avg_position` | Available at the end of March because it summarizes March search position. |
| `march_ctr` | Available at the end of March because it is calculated using March clicks and impressions. |
| `active_days` | Available at the end of March because it counts March dates with usable GSC data. |

**Label:** `position_worsened` is calculated using April average position compared with March average position. It is a future outcome and must never be used as a feature.

**Feature frame:** 158,549 content items with five features, two context identifiers, and one label. No missing values were found.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

features = [
    "impressions",
    "clicks",
    "avg_position",
    "march_ctr",
    "active_days"
]

X = feature_df[features]
y = feature_df["position_worsened"]

# Use the same train/test split for both experiments
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Honest model: only March features
honest_model = DecisionTreeClassifier(max_depth=4, random_state=42)
honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)
honest_score = accuracy_score(y_test, honest_predictions)

print("Honest model accuracy:", round(honest_score, 4))

# Deliberate leakage: copy the April answer into a feature
X_leaky = X.copy()
X_leaky["leaked_label"] = y

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(max_depth=4, random_state=42)
leaky_model.fit(X_train_leaky, y_train_leaky)

leaky_predictions = leaky_model.predict(X_test_leaky)
leaky_score = accuracy_score(y_test_leaky, leaky_predictions)

print("Leaky model accuracy:", round(leaky_score, 4))

Honest model accuracy: 0.6792
Leaky model accuracy: 1.0


### Leakage Experiment Results

| Model | Accuracy |
|---|---:|
| Honest model (March features only) | 67.92% |
| Leaky model (includes April label) | 100% |

The leaky model achieved perfect accuracy because the `leaked_label` feature directly contained the target variable. This is an example of data leakage, where information from the outcome period enters the model's features.

The honest model's 67.92% accuracy is the more realistic result because it uses only information available by the end of March.

**Conclusion:** Future information must never be included in model features. A perfect score can indicate leakage rather than a genuinely effective model.

In [10]:
X_leaky = X_leaky.drop(columns=["leaked_label"])

print("Leaked feature removed:", "leaked_label" not in X_leaky.columns)
print("Final model features:", X.columns.tolist())

Leaked feature removed: True
Final model features: ['impressions', 'clicks', 'avg_position', 'march_ctr', 'active_days']


### Deliberate Data Leakage Experiment

I trained two decision tree models using the same train-test split.

- **Honest model accuracy: 67.92%**  
  This model used only features available during March 2026.

- **Leaky model accuracy: 100%**  
  I deliberately added `position_worsened`, the target label derived from April data, as an input feature.

The leaky model achieved perfect accuracy because it was given the answer it was supposed to predict. This demonstrates how data leakage can create misleadingly high model performance.

I removed the leaked feature after the experiment. The final feature set contains only the five legitimate March features.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Limitations of This Dataset

1. **Uneven data availability:** Only 3,611,061 out of 9,841,378 March rows had usable GSC data. Only 47 of the 55 clients had GSC data available.

2. **Selected content items:** The feature frame contains 158,549 content items with usable March and April data. Items without valid GSC data or positive impressions were excluded, so the results may not represent all content.

3. **Short prediction window:** The label measures whether average search position worsened from March to April 2026. It does not establish a long-term ranking decline.

4. **Observational data:** Search position can be influenced by factors such as search intent, competition, and query mix. The model cannot establish that any feature caused a ranking change.

5. **No minimum exposure threshold:** Some content items have very few active days or impressions, which may make their average position less reliable.

### Self-check

- [x] Defined the unit of analysis and observation window.
- [x] Identified features, label, context, and excluded fields.
- [x] Executed three verification queries for grain, row count/date range, and GSC availability.
- [x] Used March 2026 for features and April 2026 for the outcome.
- [x] Kept the five features separate from the future label.
- [x] Demonstrated data leakage and removed the leaked feature.
- [x] Documented important dataset limitations.
- [x] Kept June 2026 reserved as the sealed test month.